In [1]:
import pandas as pd
import numpy as np

import tensorflow as tf

from tensorflow.keras.models import Model

from tensorflow.keras.layers import (
    Input,
    Embedding,
    Dense,
    Flatten,
    Concatenate,
    MultiHeadAttention,
    GlobalAveragePooling1D
)

from tensorflow.keras.preprocessing.sequence import pad_sequences

from sklearn.model_selection import train_test_split

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

print(tf.__version__)

2.21.0


In [2]:
route_dataset_df = pd.read_pickle(
    "../data/processed/route_dataset.pkl"
)

print(route_dataset_df.shape)
route_dataset_df.head()

(19647, 15)


,Route ID,Driver ID,Country,Week ID,Day of Week,Planned Route,Actual Route,Planned Stop Count,Actual Stop Count,Planned Distance,Actual Distance,RQS,EDS,DDS,Label
0,0,0,1,0,Monday,"[0, 1, 2, 3, 3, 0, 1]","[0, 3, 3, 0, 1, 2, 1]",7,7,49.468094,44.965197,0.5000,0.5714,-0.0910,D
1,1,1,1,0,Monday,"[0, 4, 5, 6, 7, 8, 9]","[0, 9, 4, 7, 6, 5, 8]",7,7,33.274342,33.610418,0.5000,0.5714,0.0101,D
2,2,2,1,0,Monday,"[0, 10, 11, 12, 13, 14, 15]","[0, 12, 13, 11, 10, 14, 15]",7,7,12.124804,12.508786,0.6667,0.5714,0.0317,D
3,3,3,1,0,Monday,"[0, 16, 17, 18, 19, 20, 21, 22, 20, 23]","[0, 16, 19, 17, 18, 20, 22, 20, 21, 23]",10,10,19.039848,19.374644,0.8400,0.4000,0.0176,D
4,4,4,1,0,Monday,"[0, 24, 25, 26, 27, 28, 29, 30]","[0, 24, 30, 26, 27, 28, 29, 25]",8,8,20.632674,19.528799,0.6875,0.2500,-0.0535,D


In [3]:
print(type(route_dataset_df.iloc[0]["Planned Route"]))
print(route_dataset_df.iloc[0]["Planned Route"])

<class 'list'>
[0, 1, 2, 3, 3, 0, 1]


In [4]:
MAX_ROUTE_LENGTH = route_dataset_df[
    "Planned Route"
].apply(len).max()

print(MAX_ROUTE_LENGTH)

36


In [5]:
route_sequences = pad_sequences(
    route_dataset_df["Planned Route"],
    maxlen=MAX_ROUTE_LENGTH,
    padding="post",
    truncating="post",
    value=0
)

print(route_sequences.shape)

(19647, 36)


In [6]:
max_stop_id = max([
    max(route)
    for route in route_dataset_df["Planned Route"]
])

VOCAB_SIZE = max_stop_id + 1

print(VOCAB_SIZE)

10864


In [7]:
driver_input = route_dataset_df[
    "Driver ID"
].values

numerical_features = route_dataset_df[
    [
        "Country",
        "Week ID",
        "Planned Stop Count",
        "Planned Distance"
    ]
].values

y = route_dataset_df[
    "RQS"
].values

In [8]:
(
    route_train,
    route_test,

    driver_train,
    driver_test,

    num_train,
    num_test,

    y_train,
    y_test

) = train_test_split(

    route_sequences,
    driver_input,
    numerical_features,
    y,

    test_size=0.2,
    random_state=42

)

In [9]:
# -------------------------
# Driver Branch
# -------------------------

driver_layer = Input(
    shape=(1,),
    name="Driver_Input"
)

driver_embedding = Embedding(
    input_dim=route_dataset_df["Driver ID"].max() + 1,
    output_dim=32
)(
    driver_layer
)

driver_embedding = Flatten()(
    driver_embedding
)


# -------------------------
# Route Branch
# -------------------------

route_layer = Input(
    shape=(MAX_ROUTE_LENGTH,),
    name="Route_Input"
)

route_embedding = Embedding(
    input_dim=VOCAB_SIZE,
    output_dim=64,
    mask_zero=False
)(
    route_layer
)

attention_output = MultiHeadAttention(
    num_heads=4,
    key_dim=16
)(
    route_embedding,
    route_embedding
)

route_attention = GlobalAveragePooling1D()(
    attention_output
)


# -------------------------
# Numerical Branch
# -------------------------

numerical_layer = Input(
    shape=(4,),
    name="Numerical_Input"
)

numerical_dense = Dense(
    32,
    activation="relu"
)(
    numerical_layer
)


# -------------------------
# Merge
# -------------------------

merged = Concatenate()(
    [
        driver_embedding,
        route_attention,
        numerical_dense
    ]
)

dense1 = Dense(
    128,
    activation="relu"
)(
    merged
)

dense2 = Dense(
    64,
    activation="relu"
)(
    dense1
)

output = Dense(
    1,
    activation="sigmoid"
)(
    dense2)


model = Model(
    inputs=[
        driver_layer,
        route_layer,
        numerical_layer
    ],
    outputs=output
)

In [10]:
model.compile(
    optimizer="adam",
    loss="mse",
    metrics=["mae"]
)

model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ Route_Input         │ (None, 36)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Driver_Input        │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_1         │ (None, 36, 64)    │    695,296 │ Route_Input[0][0] │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding           │ (None, 1, 32)     │     12,800 │ Driver_Input[0][… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 36, 64)    │     16,640 │ embedding_1[0][0… │
│ (MultiHeadAttentio… │                   │            │ embedding_1[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Numerical_Input     │ (None, 4)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten (Flatten)   │ (None, 32)        │          0 │ embedding[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 64)        │          0 │ multi_head_atten… │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 32)        │        160 │ Numerical_Input[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 128)       │          0 │ flatten[0][0],    │
│ (Concatenate)       │                   │            │ global_average_p… │
│                     │                   │            │ dense[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 128)       │     16,512 │ concatenate[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 64)        │      8,256 │ dense_1[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_3 (Dense)     │ (None, 1)         │         65 │ dense_2[0][0]     │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 749,729 (2.86 MB)

 Trainable params: 749,729 (2.86 MB)

 Non-trainable params: 0 (0.00 B)

In [11]:
history = model.fit(

    [
        driver_train,
        route_train,
        num_train
    ],

    y_train,

    validation_split=0.2,

    epochs=20,

    batch_size=512,

    verbose=1

)

Epoch 1/20
25/25 ━━━━━━━━━━━━━━━━━━━━ 3s 66ms/step - loss: 0.0417 - mae: 0.1551 - val_loss: 0.0374 - val_mae: 0.1555
Epoch 2/20
25/25 ━━━━━━━━━━━━━━━━━━━━ 1s 58ms/step - loss: 0.0342 - mae: 0.1438 - val_loss: 0.0311 - val_mae: 0.1365
Epoch 3/20
25/25 ━━━━━━━━━━━━━━━━━━━━ 2s 61ms/step - loss: 0.0282 - mae: 0.1313 - val_loss: 0.0247 - val_mae: 0.1255
Epoch 4/20
25/25 ━━━━━━━━━━━━━━━━━━━━ 1s 57ms/step - loss: 0.0244 - mae: 0.1178 - val_loss: 0.0235 - val_mae: 0.1206
Epoch 5/20
25/25 ━━━━━━━━━━━━━━━━━━━━ 1s 57ms/step - loss: 0.0232 - mae: 0.1133 - val_loss: 0.0223 - val_mae: 0.1125
Epoch 6/20
25/25 ━━━━━━━━━━━━━━━━━━━━ 1s 58ms/step - loss: 0.0229 - mae: 0.1125 - val_loss: 0.0238 - val_mae: 0.1108
Epoch 7/20
25/25 ━━━━━━━━━━━━━━━━━━━━ 1s 53ms/step - loss: 0.0221 - mae: 0.1088 - val_loss: 0.0211 - val_mae: 0.1082
Epoch 8/20
25/25 ━━━━━━━━━━━━━━━━━━━━ 1s 53ms/step - loss: 0.0208 - mae: 0.1048 - val_loss: 0.0206 - val_mae: 0.1043
Epoch 9/20
25/25 ━━━━━━━━━━━━━━━━━━━━ 1s 55ms/step - loss: 0.018

In [12]:
y_pred = model.predict(
    [
        driver_test,
        route_test,
        num_test
    ]
)

123/123 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step


In [13]:
import numpy as np

mae = mean_absolute_error(
    y_test,
    y_pred
)

mse = mean_squared_error(
    y_test,
    y_pred
)

rmse = np.sqrt(
    mse
)

r2 = r2_score(
    y_test,
    y_pred
)

print("="*60)
print("ATTENTION REGRESSION RESULTS")
print("="*60)

print(f"MAE  : {mae:.4f}")
print(f"RMSE : {rmse:.4f}")
print(f"R²   : {r2:.4f}")

ATTENTION REGRESSION RESULTS
MAE  : 0.1133
RMSE : 0.1641
R²   : 0.2522


In [14]:
model.save(
    "../outputs/models/attention_regression.keras"
)

print(
    "Attention Regression model saved successfully!"
)

Attention Regression model saved successfully!
